### etd_rk4.ipynb
*Created June 16, 2026* <br/> 
This notebook implements the ETD-RK4 method, a fourth-order exponential time differencing (ETD) method for solving stiff ODE systems. 



In [1]:
using LinearAlgebra, NBInclude, UnPack, Random, Printf
@nbinclude("../../../phi_functions/phi_functions.ipynb")

In [3]:
function etd_rk4(A, f, u0, tspan::NTuple{2,Float64}, p = nothing; dt::Float64)
    """
    Solves ODE system of the form du/dt = Au + f(u,p,t) for t ∈ [tspan[1],tspan[2]]
    using a fourth-order exponential time differencing (ETD) scheme. 
    
    PARAMETERS
    ----------
    A :: N x N matrix (or scalar) 
    f :: nonlinear function of the form f(u,p,t)
    u0 :: initial condition (length N vector, or a scalar)
    tspan :: time interval over which to integrate
    Δt :: time step (fixed) 
    p :: parameter for nonlinear function (if required)

    RETURNS
    -------
    sol = (u = u, t = t, p = p, Δt = Δt)

        sol.u :: vector of the solution iterates 
        sol.t :: vector of the time values corresponding to solution iterates
        Δt    :: the time step size 
        p     :: parameter for f 
    """

    #If A is a scalar, u0 must be a scalar.  If A is a matrix, u0 must be a vector. 
    (A isa AbstractMatrix && u0 isa AbstractVector) || (A isa Number && u0 isa Number) || 
    throw(ArgumentError("Types of A and u0 are incompatible: typeof(A) = $(typeof(A)), typeof(u0) = $(typeof(u0))"))

    #If A is a matrix, it must be square and u0 must have compatible size 
    if A isa AbstractMatrix
        size(A, 1) == size(A, 2) || throw(ArgumentError("A must be a square matrix. Passed A with size $(size(A))"))
        size(A, 1) == length(u0) || throw(ArgumentError("Must have size(A,1) == size(A,2) = length(u0)." ))        
    end 

    tspan[1] < tspan[2] || throw(ArgumentError("t0 must be smaller than tf. Passed t0 = $(tspan[1]), tf = $(tspan[2])."))

    
    t0, tf = tspan
    nsteps = Int(round((tf - t0) / Δt))
    t = collect(range(t0, step = Δt, length = nsteps + 1))
    u = [zero(u0) for _ in 1:nsteps + 1]
    u[1] = u0

    #Precompute some matrices 
    Z = A*Δt
    Z_half = 0.5*A*Δt

    Φ₀, Φ₁, Φ₂, Φ₃ = phis(Z, 3)
    Φ₀_half, Φ₁_half = phis(Z_half, 1)

    Φ₀ = phi(Z, 0)
    Φ₁ = phi(Z, 1)
    Φ₂ = phi(Z, 2)
    Φ₃ = phi(Z, 3)

    Φ₀_half = phi(Z_half, 0)
    Φ₁_half = phi(Z_half, 1)
    
    B₁ =   Φ₁ -  3*Φ₂ + 4*Φ₃
    B₂ =         2*Φ₂ - 4*Φ₃
    B₃ =         2*Φ₂ - 4*Φ₃   
    B₄ =      -  1*Φ₂ + 4*Φ₃

    for n = 1:nsteps #compute u[2] through u[nsteps + 1] 

        #Initial nonlinear eval
        Fn = f(u[n], p, t[n]) 

        #Half-step prediction
        a = Φ₀_half * u[n]   +   0.5 * Δt * Φ₁_half * Fn
        Fa = f(a, p, t[n] + Δt/2)

        #Second half-step prediction
        b = Φ₀_half * u[n]   +   0.5 * Δt * Φ₁_half * Fa
        Fb = f(b, p, t[n] + Δt/2)

        #Full-step prediction 
        c = Φ₀_half * a      +   0.5 * Δt * Φ₁_half * (2*Fb - Fn)
        Fc = f(c, p, t[n] + Δt)

        #Final update 
        u[n+1] = Φ₀ * u[n]   +   Δt * B₁ * Fn  +  Δt * B₂ * (Fa + Fb)  +  Δt * B₄ * Fc 
    end 

    return (u = u, t = t, Δt = Δt, p = p)
end 

etd_rk4 (generic function with 2 methods)